# Skin Analysis

This notebook performs skin analysis including extracting the skin mask (ignoring eyes/lips), evaluating skin tone (redness/dark circles), and detecting texture/blemishes.

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['figure.figsize'] = (10, 8)

base_options = mp_python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(base_options=base_options,
                                       output_face_blendshapes=True,
                                       output_facial_transformation_matrixes=True,
                                       num_faces=1)
detector = vision.FaceLandmarker.create_from_options(options)

def show_img(title, img, cmap=None):
    plt.figure()
    plt.title(title)
    if cmap:
        plt.imshow(img, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()


In [ ]:
import tkinter as tk
from tkinter import filedialog

# Create a pop-up file dialog to select an image
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
image_path = filedialog.askopenfilename(title="Select Face Image", filetypes=[("Image files", "*.jpg *.jpeg *.png")])

if image_path:
    print(f"Selected image: {image_path}")
    img = cv2.imread(image_path)
    h, w, _ = img.shape
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    detection_result = detector.detect(mp_image)

    if detection_result.face_landmarks:
        landmarks = detection_result.face_landmarks[0]
        print("Face detected.")
    else:
        print("No face detected.")
else:
    print("No image selected.")


In [ ]:
# We will create a hull of the face, and subtract the eyes, lips, and eyebrows to get pure skin.
from scipy.spatial import ConvexHull

# Feature indices from MediaPipe
FACE_OVAL = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
LIPS = [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95]
LEFT_EYE = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
RIGHT_EYE = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
LEFT_EYEBROW = [70, 63, 105, 66, 107, 55, 65, 52, 53, 46]
RIGHT_EYEBROW = [300, 293, 334, 296, 336, 285, 295, 282, 283, 276]

def get_pts(indices):
    return np.array([(int(landmarks[i].x * w), int(landmarks[i].y * h)) for i in indices])

face_pts = get_pts(FACE_OVAL)
lips_pts = get_pts(LIPS)
l_eye_pts = get_pts(LEFT_EYE)
r_eye_pts = get_pts(RIGHT_EYE)
l_brow_pts = get_pts(LEFT_EYEBROW)
r_brow_pts = get_pts(RIGHT_EYEBROW)

# Create masks
face_mask = np.zeros((h, w), dtype=np.uint8)
cv2.fillPoly(face_mask, [face_pts], 255)

exclude_mask = np.zeros((h, w), dtype=np.uint8)
for pts in [lips_pts, l_eye_pts, r_eye_pts, l_brow_pts, r_brow_pts]:
    # convex hull to ensure we fully exclude the feature
    hull = cv2.convexHull(pts)
    cv2.fillConvexPoly(exclude_mask, hull, 255)

# Subtract features from face to get skin
skin_mask = cv2.bitwise_and(face_mask, cv2.bitwise_not(exclude_mask))

skin_img = cv2.bitwise_and(img, img, mask=skin_mask)
show_img("Extracted Skin", skin_img)


In [ ]:
# Convert to LAB color space. The a channel represents Green-Red (higher is redder)
lab_img = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab_img)

# Apply mask
a_skin = cv2.bitwise_and(a, a, mask=skin_mask)

# Average redness in the skin
mean_a = np.mean(a[skin_mask == 255])
print(f"Average Skin Redness (a channel): {mean_a:.2f}")

# Highlight high redness areas (e.g. acne, rosacea, inflammation)
_, red_mask = cv2.threshold(a_skin, mean_a + 15, 255, cv2.THRESH_BINARY)
red_spots = cv2.bitwise_and(img, img, mask=red_mask)

show_img("Redness Hotspots", red_spots)


In [ ]:
# We use the Laplacian to find high frequency textures (pores, wrinkles, spots)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
laplacian = cv2.Laplacian(gray, cv2.CV_64F)
laplacian = np.abs(laplacian)
laplacian = np.uint8(np.clip(laplacian, 0, 255))

texture_skin = cv2.bitwise_and(laplacian, laplacian, mask=skin_mask)

mean_texture = np.mean(laplacian[skin_mask == 255])
print(f"Average Skin Texture/Roughness: {mean_texture:.2f}")

# Highlight rough patches
_, rough_mask = cv2.threshold(texture_skin, 30, 255, cv2.THRESH_BINARY)
rough_spots = cv2.bitwise_and(img, img, mask=rough_mask)

show_img("Texture/Blemishes", texture_skin, cmap='gray')


In [ ]:
# Define under eye regions
L_UNDER_EYE = [110, 205, 50, 207, 214, 192, 212]
R_UNDER_EYE = [339, 425, 280, 427, 434, 416, 432]

l_under_pts = get_pts(L_UNDER_EYE)
r_under_pts = get_pts(R_UNDER_EYE)

under_eye_mask = np.zeros((h, w), dtype=np.uint8)
cv2.fillPoly(under_eye_mask, [cv2.convexHull(l_under_pts)], 255)
cv2.fillPoly(under_eye_mask, [cv2.convexHull(r_under_pts)], 255)

# Calculate Luminance (L channel from LAB) in under eye vs rest of cheeks
mean_l_under_eye = np.mean(l[under_eye_mask == 255])

# Compare with average face luminance
mean_l_face = np.mean(l[skin_mask == 255])

print(f"Average Face Luminance: {mean_l_face:.2f}")
print(f"Under-Eye Luminance: {mean_l_under_eye:.2f}")

if mean_l_under_eye < mean_l_face * 0.9:
    print("Dark circles detected (significantly darker than surrounding skin).")
else:
    print("No prominent dark circles detected.")

under_eye_img = cv2.bitwise_and(img, img, mask=under_eye_mask)
show_img("Under Eye Region", under_eye_img)


In [ ]:
# Initialize final metrics dictionary
qoves_metrics = {}


In [ ]:
# --- Skin Undertone ---
# b channel (yellow/blue), a channel (red/green)
mean_b = np.mean(b[skin_mask == 255])
# Neutral gray in LAB is ~128. 
b_offset = mean_b - 128
a_offset = mean_a - 128

if b_offset > 10 and a_offset < 10:
    undertone = "Warm"
elif b_offset > 5 and a_offset < 5:
    undertone = "Neutral-Warm"
elif a_offset > 10 and b_offset < 5:
    undertone = "Cool"
elif a_offset > 5:
    undertone = "Neutral-Cool"
else:
    undertone = "Neutral"

qoves_metrics['undertone'] = undertone
print(f"Skin Undertone: {undertone} (a*: {mean_a:.1f}, b*: {mean_b:.1f})")


In [ ]:
# --- Skin Roughness (RIN) ---
# Map laplacian variance to a 0.05 - 0.25 RIN scale
# Variance typically ranges from 100 (smooth) to 1500 (rough) in laplacian
gray_skin = cv2.bitwise_and(gray, gray, mask=skin_mask)
variance = np.var(laplacian[skin_mask == 255])

# Mathematical mapping: 0.08 (smooth) to 0.18 (rough)
# Normalizing factor based on empirical observation of 8-bit images
rin_roughness = np.clip(0.05 + (variance / 3000.0), 0.05, 0.30)
qoves_metrics['roughness_rin'] = rin_roughness

if rin_roughness < 0.10:
    rough_class = "Smooth"
elif rin_roughness < 0.14:
    rough_class = "Slightly Textured"
else:
    rough_class = "Textured/Rough"

qoves_metrics['texture'] = rough_class
print(f"Skin Roughness: {rin_roughness:.2f} RIN ({rough_class})")


In [ ]:
# --- Skin Oiliness (Skewness) ---
from scipy.stats import skew

# Analyze the L channel (luminance) of the skin
l_pixels = l[skin_mask == 255]

# Specular highlights (shine from oil) stretch the tail of the histogram to the right (positive skew)
# A very matte face will have negative or zero skew. Oily face > 0.1 skew.
luminance_skew = skew(l_pixels)
qoves_metrics['oiliness_skew'] = luminance_skew

if luminance_skew > 0.30:
    oil_class = "Oily/Shiny"
elif luminance_skew > 0.0:
    oil_class = "Normal/Combination"
else:
    oil_class = "Matte/Dry"
    
print(f"Skin Oiliness: {luminance_skew:.2f} skewness ({oil_class})")


In [ ]:
# --- Skin Homogeneity (RIN) ---
# Homogeneity is the inverse of standard deviation of color (unevenness)
std_l = np.std(l[skin_mask == 255])
std_a = np.std(a[skin_mask == 255])
std_b = np.std(b[skin_mask == 255])

# Combine standard deviations
total_std = (std_l + std_a + std_b) / 3.0

# Map to Homogeneity RIN (0.15 - 0.40 range)
# Higher standard deviation = HIGHER RIN (More uneven)
homogeneity_rin = np.clip(0.10 + (total_std / 50.0), 0.10, 0.45)
qoves_metrics['homogeneity_rin'] = homogeneity_rin

if homogeneity_rin < 0.20:
    homo_class = "Even"
elif homogeneity_rin < 0.28:
    homo_class = "Slightly Uneven"
else:
    homo_class = "Uneven"

qoves_metrics['evenness'] = homo_class
print(f"Skin Homogeneity: {homogeneity_rin:.2f} RIN ({homo_class})")


In [ ]:
# --- Skin Blemishing ---
# Count connected components in the red spots mask we generated earlier
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(red_mask, connectivity=8)

# Filter out tiny noise (area < 5) and huge areas (diffuse redness, not a blemish, area > 500)
blemish_count = 0
for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]
    if 5 < area < 500:
        blemish_count += 1

qoves_metrics['blemish_count'] = blemish_count

if blemish_count < 3:
    blem_class = "Clear"
elif blemish_count < 10:
    blem_class = "Mild"
elif blemish_count < 25:
    blem_class = "Moderate"
else:
    blem_class = "Severe"

qoves_metrics['blemishing'] = blem_class
print(f"Skin Blemishing: {blemish_count} distinct spots detected ({blem_class})")


In [ ]:
# --- FINAL QOVES-STYLE SUMMARY ---
import json

summary = {
    "Summary of your skin": {
        "SKIN UNDERTONE": qoves_metrics['undertone'],
        "SKIN BLEMISHING": qoves_metrics['blemishing'],
        "SKIN EVENNESS": qoves_metrics['evenness'],
        "SKIN TEXTURE": qoves_metrics['texture']
    },
    "Scientific Metrics": {
        "SKIN ROUGHNESS": f"{qoves_metrics['roughness_rin']:.2f} RIN",
        "SKIN OILINESS": f"{qoves_metrics['oiliness_skew']:.2f} skewness",
        "SKIN HOMOGENEITY": f"{qoves_metrics['homogeneity_rin']:.2f} RIN"
    }
}

print("\n" + "="*50)
print("QOVES SKIN ANALYSIS REPORT")
print("="*50)
print(json.dumps(summary, indent=4))


In [ ]:
# --- WRINKLE ANALYSIS ---
# Calculate wrinkle depth (mm) programmatically using OpenCV

import cv2
import numpy as np
import ipywidgets as widgets
import base64
from IPython.display import display, HTML, clear_output
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image

def image_to_base64(img):
    if len(img.shape) == 3 and img.shape[2] == 4:
        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGBA2BGRA)
    else:
        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.png', img_bgr)
    return base64.b64encode(buffer).decode('utf-8')

# Ensure detector is available (from previous cells)
if 'detector' not in locals():
    from mediapipe.tasks.python import vision
    from mediapipe.tasks.python.core import base_options
    options = vision.FaceLandmarkerOptions(
        base_options=base_options.BaseOptions(model_asset_path='c:/AI-Face-Analysis/face_landmarker.task'),
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        num_faces=1)
    detector = vision.FaceLandmarker.create_from_options(options)

root = tk.Tk()
root.attributes('-topmost', True)
root.withdraw()
messagebox.showinfo('Image Upload', 'Please upload a FRONT face image for Wrinkle Analysis.')
img_path = filedialog.askopenfilename(title='Select Front Face Image', filetypes=[('Image files', '*.jpg *.jpeg *.png')])
root.destroy()

if not img_path:
    print("Image selection cancelled.")
else:
    img = np.array(Image.open(img_path).convert('RGB'))
    ih, iw, _ = img.shape
    
    import mediapipe as mp
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(img))
    res = detector.detect(mp_img)
    
    if not res.face_landmarks:
        print("No face detected!")
    else:
        lms = res.face_landmarks[0]
        def get_pt(idx): return np.array([int(lms[idx].x * iw), int(lms[idx].y * ih)])
        
        IPD_MM = 63.0
        left_pupil = get_pt(468)
        right_pupil = get_pt(473)
        ipd_px = np.linalg.norm(left_pupil - right_pupil)
        mm_per_px = IPD_MM / ipd_px
        
        regions = {
            'Forehead': [103, 67, 109, 10, 338, 297, 332, 284, 251, 389, 356, 454],
            'Nose': [168, 6, 197, 195, 5, 4, 1, 19, 94],
            'L. Undereye': [111, 117, 118, 119, 120, 121, 128, 245],
            'R. Undereye': [340, 346, 347, 348, 349, 350, 357, 465],
            'L. Cheek': [205, 206, 216, 212, 210, 211, 214, 192],
            'R. Cheek': [425, 426, 436, 432, 430, 431, 434, 416],
            'Chin': [152, 148, 176, 149, 150, 136, 172, 58, 132, 215, 435, 361]
        }
        
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced_gray = clahe.apply(gray)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        blackhat = cv2.morphologyEx(enhanced_gray, cv2.MORPH_BLACKHAT, kernel)
        
        region_data = {}
        overall_score_accum = 0
        
        for name, indices in regions.items():
            pts = np.array([get_pt(idx) for idx in indices], np.int32)
            if name == 'Forehead':
                y_shift = int(30 / mm_per_px)
                pts[:, 1] -= y_shift
                
            mask = np.zeros((ih, iw), dtype=np.uint8)
            hull = cv2.convexHull(pts)
            cv2.fillConvexPoly(mask, hull, 255)
            
            masked_bh = cv2.bitwise_and(blackhat, blackhat, mask=mask)
            max_val = np.percentile(masked_bh[mask > 0], 95) if np.sum(mask) > 0 else 0
            
            depth_mm = (max_val * 0.0015) + 0.05
            overall_score_accum += max_val
            
            heatmap_color = cv2.applyColorMap(masked_bh, cv2.COLORMAP_INFERNO)
            alpha = (masked_bh.astype(np.float32) / 255.0) * 1.5
            alpha = np.clip(alpha, 0, 1)
            
            overlay = img.copy()
            for c in range(3):
                overlay[:,:,c] = np.where(mask > 0, 
                                          overlay[:,:,c] * (1 - alpha) + heatmap_color[:,:,c] * alpha,
                                          overlay[:,:,c])
                                          
            img_str = image_to_base64(overlay)
            region_data[name] = {'depth': depth_mm, 'image': img_str}
            
        avg_intensity = overall_score_accum / len(regions)
        score = max(0, min(100, 100 - (avg_intensity * 0.5)))
        score = int(score)
        if score >= 80: score_lbl = "Good"
        elif score >= 60: score_lbl = "Average"
        else: score_lbl = "Needs Attention"
        score_color = "#166534" if score >= 80 else "#9A3412" if score < 60 else "#B45309"
        score_bg = "#F0FDF4" if score >= 80 else "#FFF7ED" if score < 60 else "#FEF3C7"
        
        region_selector = widgets.ToggleButtons(options=list(regions.keys()), button_style='')
        out = widgets.Output()
        
        def get_explanation(region, depth):
            if depth < 0.15:
                if region == 'Forehead': return "You show only very faint expression lines on the forehead so structural aging remains minimal in this zone."
                elif region == 'Nose': return "The nose shows no fixed wrinkles so the skin here retains youthful elasticity."
                elif region == 'Chin': return "The chin shows no fixed lines or mental crease so structural aging remains minimal here."
                else: return f"The {region.lower()} area shows only faint fine lines reflecting normal micro-texture rather than deep structural folding."
            else:
                if region == 'Forehead': return "You have visible horizontal lines forming across the forehead due to regular expression and structural folding."
                elif region == 'Nose': return "Bunny lines are becoming visible on the bridge of the nose."
                elif region == 'Chin': return "A slight mental crease is developing above the chin."
                else: return f"You display visible folding or depth in the {region.lower()} area which is a natural part of structural skin aging."

        def update_dashboard(change=None):
            with out:
                clear_output(wait=True)
                sel = region_selector.value
                data = region_data[sel]
                d = data['depth']
                img_b64 = data['image']
                exp = get_explanation(sel, d)
                
                max_depth_px = 60
                dip = min(max_depth_px, (d / 0.5) * max_depth_px)
                
                html = f"""
                <style>
                    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
                    .w-dash {{ font-family: 'Inter', sans-serif; color: #2D3748; max-width: 1000px; margin: 0 auto; background: white; padding-bottom: 40px; }}
                    .w-title {{ font-size: 32px; font-weight: 600; margin-bottom: 8px; letter-spacing: -0.5px; margin-top: 40px; }}
                    .w-title span {{ color: #A0AEC0; }}
                    .w-sub {{ font-size: 15px; color: #718096; margin-bottom: 32px; }}
                    .w-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 32px; }}
                    .w-left {{ position: relative; border-radius: 16px; overflow: hidden; display: flex; align-items: center; justify-content: center; }}
                    .w-left img {{ width: 100%; height: 100%; object-fit: cover; border-radius: 16px; }}
                    .float-score {{ position: absolute; bottom: 0; left: 0; width: 100%; background: white; border-top: 1px solid #F1F5F9; border-top-left-radius: 16px; border-top-right-radius: 16px; padding: 24px; box-shadow: 0 -4px 6px -1px rgba(0, 0, 0, 0.05); }}
                    .fs-top {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; }}
                    .fs-lbl {{ font-size: 14px; font-weight: 600; color: #1A202C; }}
                    .fs-tag {{ background: {score_bg}; color: {score_color}; font-size: 12px; font-weight: 500; padding: 4px 12px; border-radius: 12px; }}
                    .fs-val-wrap {{ display: flex; align-items: flex-end; }}
                    .fs-val {{ font-size: 48px; font-weight: 400; line-height: 1; letter-spacing: -1px; }}
                    .fs-max {{ font-size: 16px; color: #A0AEC0; margin-bottom: 6px; margin-left: 4px; }}
                    .w-right {{ display: flex; flex-direction: column; gap: 20px; }}
                    .panel-box {{ border: 1px solid #F1F5F9; border-radius: 16px; padding: 24px; }}
                    .pb-lbl {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1.5px; color: #A0AEC0; font-weight: 600; margin-bottom: 16px; }}
                    .depth-row {{ display: flex; justify-content: space-between; align-items: stretch; }}
                    .depth-val {{ font-size: 36px; font-weight: 500; color: #1A202C; display: flex; align-items: baseline; gap: 8px; }}
                    .depth-unit {{ font-size: 20px; font-weight: 400; color: #4A5568; }}
                    .v-scale {{ width: 6px; background: #CBD5E1; border-radius: 3px; position: relative; display: flex; flex-direction: column; height: 120px; }}
                    .v-fill {{ position: absolute; top: 0; width: 100%; background: #94A3B8; border-radius: 3px; }}
                    .graph-container {{ position: relative; width: 100%; height: 200px; background: #FAFAFA; border: 1px solid #F1F5F9; border-radius: 8px; margin-top: 16px; overflow: hidden; }}
                </style>
                <div class="w-dash">
                    <div class="w-title">An analysis of your <span>wrinkles</span></div>
                    <div class="w-sub">We analyze the average predicted <b>depth</b> of wrinkles along the face.</div>
                    <div class="w-grid">
                        <div class="w-left">
                            <img src="data:image/png;base64,{img_b64}" />
                            <div class="float-score">
                                <div class="fs-top">
                                    <div class="fs-lbl">Wrinkle Score</div>
                                    <div class="fs-tag">{score_lbl}</div>
                                </div>
                                <div class="fs-val-wrap">
                                    <div class="fs-val">{score}</div>
                                    <div class="fs-max">/100</div>
                                </div>
                            </div>
                        </div>
                        <div class="w-right">
                            <div class="panel-box" style="position: relative;">
                                <div class="pb-lbl">WRINKLE DEPTH</div>
                                <div class="depth-row">
                                    <div class="depth-val">{d:.2f} <span class="depth-unit">mm</span></div>
                                    <div style="position: relative; width: 150px; display: flex; justify-content: flex-end;">
                                        <div style="position: absolute; right: 20px; top: {min(100, (d/1.0)*100)}%; transform: translateY(-50%); font-size: 10px; font-weight: bold; background: white; padding: 2px 6px; border: 1px solid #E2E8F0; border-radius: 8px;">You ({d:.2f}mm)</div>
                                        <div class="v-scale">
                                            <div class="v-fill" style="height: {min(100, (d/1.0)*100)}%;"></div>
                                        </div>
                                    </div>
                                </div>
                            </div>
                            <div class="panel-box">
                                <div style="display: flex; justify-content: space-between; font-size: 11px; font-weight: 600; color: #718096; margin-bottom: 8px;">
                                    <span>Outer Corner</span><span>Mid</span><span>Inner Corner</span>
                                </div>
                                <div class="graph-container">
                                    <svg width="100%" height="100%" viewBox="0 0 400 200" preserveAspectRatio="none">
                                        <line x1="0" y1="50" x2="400" y2="50" stroke="#CBD5E1" stroke-width="1" stroke-dasharray="4 4" />
                                        <line x1="0" y1="100" x2="400" y2="100" stroke="#CBD5E1" stroke-width="1" stroke-dasharray="4 4" />
                                        <line x1="0" y1="150" x2="400" y2="150" stroke="#CBD5E1" stroke-width="1" stroke-dasharray="4 4" />
                                        <line x1="200" y1="0" x2="200" y2="200" stroke="#CBD5E1" stroke-width="1" stroke-dasharray="4 4" />
                                        <path d="M0,130 Q200,{130 + dip} 400,130 L400,200 L0,200 Z" fill="url(#grad)" opacity="0.5" />
                                        <path d="M0,130 Q200,{130 + dip} 400,130" fill="none" stroke="#64748B" stroke-width="2" />
                                        <path d="M0,130 Q200,130 400,130" fill="none" stroke="#CBD5E1" stroke-width="1" stroke-dasharray="2 2" />
                                        <circle cx="200" cy="{130 + dip}" r="4" fill="#64748B" />
                                        <line x1="200" y1="130" x2="200" y2="{130 + dip}" stroke="#EF4444" stroke-width="1" />
                                        <defs>
                                            <linearGradient id="grad" x1="0%" y1="0%" x2="0%" y2="100%">
                                                <stop offset="0%" style="stop-color:#94A3B8;stop-opacity:0.4" />
                                                <stop offset="100%" style="stop-color:#E2E8F0;stop-opacity:0.1" />
                                            </linearGradient>
                                        </defs>
                                    </svg>
                                </div>
                            </div>
                            <div class="panel-box">
                                <div class="pb-lbl">EXPLANATION</div>
                                <div style="font-size: 14px; color: #4A5568; line-height: 1.6;">{exp}</div>
                            </div>
                        </div>
                    </div>
                </div>
                """
                display(HTML(html))
                
        region_selector.observe(update_dashboard, names='value')
        display(HTML("<style>.widget-toggle-button { font-family: 'Inter', sans-serif; background: white; border: 1px solid #E2E8F0; } .widget-toggle-button.mod-active { background: #F8FAFC; border-bottom: 2px solid #1A202C; font-weight: 600; color: #1A202C; }</style>"))
        display(widgets.VBox([region_selector, out]))
        update_dashboard()

